### Project Name  - **Real-Time Object Detection using YOLOv8**

#### Introduction 

Object detection is a core task in computer vision that enables machines to identify and locate objects within images and video streams instantly. This project implements object detection using YOLOv8, the latest and most advanced version of the YOLO (You Only Look Once) family developed by Ultralytics. YOLOv8 is designed for high-speed, high-accuracy detection, making it ideal for real-time applications.

The goal of this project is to build a system capable of detecting multiple objects from images, recorded videos, and live webcam feeds with minimal latency. The model processes input data in a single forward pass, enabling fast predictions while maintaining strong detection performance.

### Problem Statement

In today’s rapidly evolving technological landscape, there is a growing demand for intelligent systems capable of automatically identifying and tracking objects in real-time from images and video streams. Manual monitoring and analysis of visual data—such as surveillance footage, traffic recordings, or industrial inspection videos—are time-consuming, inefficient, and prone to human error.

The problem addressed in this project is the development of an accurate and efficient real-time object detection system using YOLOv8. The system must be capable of detecting and localizing multiple objects simultaneously in both static images and continuous video streams with high speed and precision.

**Import Necessary Libraries**

In [ ]:
import os
import random
import pandas as pd
from PIL import Image
import cv2
from ultralytics import YOLO
from IPython.display import Video
import numpy as np  
import yaml


from PIL import Image, ImageDraw
from IPython.display import display


%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
sns.set(style='darkgrid')
import pathlib
import glob
from tqdm.notebook import trange, tqdm
import warnings
warnings.filterwarnings('ignore')

**Dataset Folder Structure Verification**

In [ ]:

# Path to the main folders
main_folders = [
    "C:/Users/Seema Kukkar/OneDrive/Desktop/train",
    "C:/Users/Seema Kukkar/OneDrive/Desktop/test",
    "C:/Users/Seema Kukkar/OneDrive/Desktop/valid"
]

# Iterate through each main folder
for main_folder in main_folders:
    print(f"\nMain Folder: {main_folder}")
    
    # Subfolders under the main folder (images and labels)
    subfolders = ['images', 'labels']
    
    # Iterate through each subfolder
    for subfolder in subfolders:
        # Full path to the subfolder
        subfolder_path = os.path.join(main_folder, subfolder)
        
        # Count the number of files in the subfolder
        file_count = len(os.listdir(subfolder_path))
        
        # Print the results
        print(f"  {subfolder.capitalize()} Folder: {file_count} files")

***Loading and Extracting Class Names from YAML File***

In [ ]:
# Specify the path to the YAML file
yaml_file_path = r"C:/Users/Seema Kukkar/OneDrive/Desktop/dataset.yaml"

# Load the YAML file
with open(yaml_file_path, 'r') as file:
    data = yaml.safe_load(file)

# Extract and display the class names
class_names = data['names']
print("Class Names:", class_names)

**Random Image Inspection from Training Dataset**

In [ ]:
# Path to the images folder
images_folder = r"C:/Users/Seema Kukkar/OneDrive/Desktop/train/images"

# Get a list of all image files in the folder
image_files = [f for f in os.listdir(images_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Choose a random image from the list
random_image = random.choice(image_files)

# Construct the full path to the random image
random_image_path = os.path.join(images_folder, random_image)

# Open the image using PIL
image = Image.open(random_image_path)

# Get the size, mode (channels), and number of channels of the image
image_size = image.size
image_mode = image.mode
num_channels = image.layers if hasattr(image, 'layers') else len(image.getbands())

# Print the details of the random image
print(f"Random Image: {random_image}")
print(f"Image Size: {image_size}")
print(f"Image Mode (Channels): {image_mode}")
print(f"Number of Channels: {num_channels}")

**Data Preprocessing**

**Conversion of Bounding Box Coordinates to YOLO Normalized Format**

In [ ]:
def convert_to_yolo(size, box):
    img_w, img_h = size
    xmin, ymin, xmax, ymax = box

    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h

    return x_center, y_center, width, height


In [ ]:
x_center, y_center, width, height = convert_to_yolo( (100,  100), (20,30,60,80))

In [ ]:
x_center, y_center, width, height

**Remove Corrupt Images**

In [ ]:
import os
from PIL import Image

images_folder = r"C:/Users/Seema Kukkar/OneDrive/Desktop/train/images"

removed_count = 0


for file in os.listdir(images_folder):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        file_path = os.path.join(images_folder, file)
        try:
            img = Image.open(file_path)
            img.verify()   # verify image integrity
        except:
            print(f"❌ Removing corrupt image: {file}")
            os.remove(file_path)
            removed_count += 1

print(f"✔ Removed {removed_count} corrupt images.")

**Label–Image Matching**

In [ ]:
images_folder = r"C:/Users/Seema Kukkar/OneDrive/Desktop/train/images"
labels_folder = r"C:/Users/Seema Kukkar/OneDrive/Desktop/train/labels"

valid_images = []


for image_file in os.listdir(images_folder):
    if image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        label_file = os.path.splitext(image_file)[0] + ".txt"
        label_path = os.path.join(labels_folder, label_file)

        if os.path.exists(label_path):
            valid_images.append(image_file)
        else:
            print(f"⚠ Missing label for image: {image_file}")

print(f"✔ Total matched image-label pairs: {len(valid_images)}")


**Split Dataset into Train / Validation**

In [ ]:
import os

dataset_path = r"C:/Users/Seema Kukkar/OneDrive/Desktop/dataset"
images_path = os.path.join(dataset_path, "images")

print("Images path exists:", os.path.exists(images_path))
print("Images path:", images_path)


In [ ]:
print(os.listdir(images_path))


**Random Sample Object Detection Using Pretrained YOLOv8 Model**

In [ ]:
# Path to the images folder
images_folder = r"C:/Users/Seema Kukkar/OneDrive/Desktop/train/images"

# Get a list of all image files in the folder
image_files = [f for f in os.listdir(images_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

# Choose four random images from the list
random_images = random.sample(image_files, 4)

# Use a pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Loop through the selected images and show prediction results
for image_file in random_images:
    # Construct the full path to the image
    image_path = os.path.join(images_folder, image_file)
    
    # Use the model to detect objects
    result_predict = model.predict(source=image_path, imgsz=(416))
    
    # Display the original image
    original_image = Image.open(image_path)
    display(original_image)
    
    # Display the prediction results
    for result in result_predict:
        plot = result.plot()
        plot = cv2.cvtColor(plot, cv2.COLOR_BGR2RGB)
        display(Image.fromarray(plot))

**YOLOv8 Model Training Using Custom Dataset Configuration**

In [ ]:
results = model.train(
    data="C:/Users/Seema Kukkar/OneDrive/Desktop/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device= "cpu" 
)


**Evaluation**

In [ ]:
metrics = model.val()

print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)


**Visualization of Normalized Confusion Matrix After Model Training**

In [ ]:
from IPython.display import Image, display

# Specify the path to the image
image_path = r"C:/Users/Seema Kukkar/Downloads/YOLO V8 vehicle detection/runs/detect/train2/confusion_matrix_normalized.png"

# Display the image
display(Image(filename=image_path))

- The darker boxes on the diagonal show correct classification.
- Higher value are better prediction for that class.
- off diagonal values shows misclassifications.

In [ ]:
from IPython.display import Image, display

# Specify the path to the image
image_path = r"C:/Users/Seema Kukkar/Downloads/YOLO V8 vehicle detection/runs/detect/train2/BoxF1_curve.png"

# Display the image
display(Image(filename=image_path))

Ambulance and Bus classes perform well, while Truck shows lower performance.

In [ ]:
from IPython.display import Image, display

# Specify the path to the image
image_path = r"C:/Users/Seema Kukkar/Downloads/YOLO V8 vehicle detection/runs/detect/train2/results.png"

# Display the image
display(Image(filename=image_path))

- Loss is decreasing.
- Metrics are increasing.

**Visualization of Input Test Video for Model Evaluation**

In [ ]:
from IPython.display import Video

video_path = r"C:/Users/Seema Kukkar/OneDrive/Desktop/TrafficPolice.mp4"

Video(video_path, embed=True, width=800)


**Video Object Detection Using Trained YOLOv8 Model**

In [ ]:

model.predict(
    source= r"C:/Users/Seema Kukkar/OneDrive/Desktop/TestVideo/TrafficPolice.mp4",
    save=True,
    conf=0.3
)


In [ ]:
import cv2
from ultralytics import YOLO


input_video = r"C:/Users/Seema Kukkar/Downloads/YOLO V8 vehicle detection/runs/detect/predict/TrafficPolice.avi"

cap = cv2.VideoCapture(input_video)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Lower confidence threshold
    results = model(frame, conf=0.20)

    print("Detected objects:", results[0].boxes.cls)

    annotated_frame = results[0].plot()

    cv2.imshow("Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
